# ECoRe continuation · train scorer and build a new experimental index

This run reuses Part 7 embeddings. It evaluates the production-available language-only cohort, trains one final candidate-identity-free scorer on all dev episodes, and attaches it to a copy of the expanded index. The current production index is never overwritten.

In [ ]:
from google.colab import drive
from pathlib import Path
import json, os, subprocess, sys

drive.mount('/content/drive')
REPO = Path('/content/drive/MyDrive/style_matching')
os.chdir(REPO)
EXP = REPO / 'artifacts/source_expansion_v2'
HELDOUT = REPO / 'data/all/meta/all_source_heldout_splits.parquet'
EMBEDDINGS = EXP / 'expanded_source_heldout_eval'
BASE_INDEX = REPO / 'artifacts/multilingual_style_index_challenger_expanded_v2'
EVAL = EXP / 'ecore_language_only_v2'
NEW_INDEX = REPO / 'artifacts/multilingual_style_index_ecore_experimental_v1'
for path in (HELDOUT, EMBEDDINGS / 'style_embedding_scores.npz', BASE_INDEX / 'metadata.json'):
    assert path.exists(), f'Missing prerequisite: {path}'

def run(cmd):
    print('>>>', ' '.join(map(str, cmd)), flush=True)
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise RuntimeError(f'command failed with exit {code}: {" ".join(map(str, cmd))}')

In [ ]:
run([
    sys.executable, 'scripts/evaluate_ecore_innovations.py',
    '--input', str(HELDOUT), '--embedding-dir', str(EMBEDDINGS),
    '--output-dir', str(EVAL), '--environment-policy', 'language_only',
    '--temperature', '0.08', '--author-folds', '5', '--hard-negatives', '12',
    '--bootstrap-runs', '5000', '--train-cap', '300',
    '--embedding-seed', '20260701', '--seed', '20260725',
])
report = json.loads((EVAL / 'ecore_innovation_metrics.json').read_text())
gate = report['innovation_3_episodic_transfer']['direct_deployment_contrast_vs_centroid']
print('DIRECT ECoRe − centroid:', gate)
print('DEPLOYMENT GATE:', report['innovation_3_episodic_transfer']['deployment_gate_passed'])

In [ ]:
run([
    sys.executable, 'scripts/build_ecore_experimental_index.py',
    '--base-index', str(BASE_INDEX),
    '--scorer', str(EVAL / 'ecore_production_scorer.json'),
    '--output-dir', str(NEW_INDEX), '--overwrite',
])
metadata = json.loads((NEW_INDEX / 'metadata.json').read_text())
print('NEW INDEX:', NEW_INDEX)
print('STATUS:', metadata['score_status'])
print('DEPLOYMENT GATE PASSED:', metadata['ecore_deployment_gate_passed'])
if not metadata['ecore_deployment_gate_passed']:
    print('Index created for research comparison only. Keep the centroid index in production.')